# 04 — Model training and hyperparameter optimization
Four regression models + persistence baseline. All preprocessing lives inside scikit-learn pipelines.

| # | Model | Pipeline |
|---|---|---|
| 1 | Ridge Regression | median imputer → [selector] → StandardScaler → Ridge |
| 2 | Random Forest | median imputer → [selector] → RandomForestRegressor |
| 3 | XGBoost | median imputer → [selector] → XGBRegressor |
| 4 | SVR (RBF) | median imputer → [selector] → StandardScaler → SVR |
| – | Persistence | ŷ = trailing 24-h mean PM2.5 at 18:00 |

Hyperparameters: `RandomizedSearchCV` on the 2021–2023 training targets with `TimeSeriesSplit` (gap = 1 day),
scoring = MAE. Selection by 2024 validation MAE. Final refit on train + validation; the 2025 test is used once.

In [1]:
# --- Setup: locate the project root and import the package --------------------
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd
from IPython.display import Image, display, Markdown
from air_quality.config import load_config, setup_logging
cfg = load_config(ROOT / "config.yaml"); setup_logging(cfg, "notebooks")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)
FIG = cfg.path("figures_dir"); MET = cfg.path("metrics_dir")
print("Project root:", ROOT)

Project root: /Users/dannieey/Desktop/ML_final


In [2]:
from air_quality.models import build_pipeline, param_space, MODEL_NAMES
from air_quality.training import run_training, load_model_frame, make_cv
df = load_model_frame(cfg)
print(df.groupby("split").size())
for m in MODEL_NAMES:
    print(f"\n{m}:", build_pipeline(m, select=False))
    print("  search space:", {k: (v if isinstance(v, list) else f"{type(v).__name__}") for k, v in param_space(m).items()})
print("\nCV:", make_cv(cfg, (df.split == "train").sum()))

split
test           346
train         1061
unused           5
validation     351
dtype: int64

ridge: Pipeline(steps=[('impute',
                 SimpleImputer(keep_empty_features=True, strategy='median')),
                ('scale', StandardScaler()), ('model', Ridge())])
  search space: {'model__alpha': 'rv_continuous_frozen'}

random_forest: Pipeline(steps=[('impute',
                 SimpleImputer(keep_empty_features=True, strategy='median')),
                ('model',
                 RandomForestRegressor(n_estimators=300, n_jobs=-1,
                                       random_state=42))])
  search space: {'model__n_estimators': [200, 300, 500], 'model__max_depth': [None, 6, 10, 16], 'model__min_samples_split': [2, 5, 10], 'model__min_samples_leaf': [1, 2, 4, 8], 'model__max_features': ['sqrt', 0.33, 0.5, 1.0]}

xgboost: Pipeline(steps=[('impute',
                 SimpleImputer(keep_empty_features=True, strategy='median')),
                ('model',
                 XGBRegresso

/Users/dannieey/Desktop/ML_final/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Run training
Set `RUN_TRAINING = False` to reuse frozen results in `artifacts/` (recommended once the benchmark is frozen).

In [3]:
RUN_TRAINING = not (cfg.path("metrics_dir") / "all_results.csv").exists()
if RUN_TRAINING:
    out = run_training(cfg)          # full budget from config.yaml (n_iter per model/config)
res = pd.read_csv(MET / "all_results.csv")
print("Rows in all_results.csv:", len(res))

Rows in all_results.csv: 142


## Cross-validation and validation results (tuned models)

In [4]:
v = res[(res.stage == "validation")].pivot_table(index=["model", "feature_config"], columns="variant", values=["cv_mae", "mae", "rmse", "r2"])
display(v.round(3))

cv_mae      mae                      r2                    rmse               
variant                          tuned baseline default  tuned baseline default  tuned baseline default  tuned
model         feature_config                                                                                  
ensemble      basic                NaN      NaN   3.305  3.105      NaN   0.342  0.461      NaN   5.169  4.678
              full                 NaN      NaN   3.184  3.091      NaN   0.433  0.451      NaN   4.796  4.722
              full_aqi             NaN      NaN   3.213  3.117      NaN   0.427  0.452      NaN   4.824  4.716
              full_selected        NaN      NaN   3.133  3.070      NaN   0.448  0.463      NaN   4.731  4.669
              full_weather         NaN      NaN   2.374  2.227      NaN   0.680  0.723      NaN   3.606  3.352
              full_weather_past    NaN      NaN   3.003  2.898      NaN   0.475  0.509      NaN   4.616  4.463
              lag                  NaN      NaN   3.192  3.099      NaN   0.427  0.454      NaN   4.821  4.706
persistence   pm25_mean_24h        NaN    3.711     NaN    NaN    0.171     NaN    NaN    5.802     NaN    NaN
random_forest basic              2.999      NaN   3.394  3.116      NaN   0.321  0.449      NaN   5.248  4.729
              full               3.042      NaN   3.291  3.142      NaN   0.421  0.463      NaN   4.849  4.668
              full_aqi           3.063      NaN   3.288  3.136      NaN   0.422  0.463      NaN   4.844  4.667
              full_selected      3.024      NaN   3.242  3.094      NaN   0.434  0.476      NaN   4.792  4.610
              full_weather       2.390      NaN   2.349  2.352      NaN   0.703  0.708      NaN   3.472  3.440
              full_weather_past  2.975      NaN   3.032  2.940      NaN   0.511  0.527      NaN   4.456  4.383
              lag                3.070      NaN   3.257  3.118      NaN   0.430  0.468      NaN   4.809  4.649
ridge         basic              3.151      NaN   3.392  3.405      NaN   0.427  0.425      NaN   4.821  4.829
              full               3.162      NaN   3.248  3.225      NaN   0.450  0.465      NaN   4.724  4.658
              full_aqi           3.169      NaN   3.282  3.265      NaN   0.443  0.458      NaN   4.753  4.690
              full_selected      3.140      NaN   3.215  3.299      NaN   0.463  0.463      NaN   4.670  4.670
              full_weather       2.654      NaN   2.707  2.637      NaN   0.655  0.667      NaN   3.744  3.675
              full_weather_past  3.078      NaN   3.120  3.065      NaN   0.502  0.515      NaN   4.495  4.439
              lag                3.220      NaN   3.293  3.342      NaN   0.452  0.448      NaN   4.716  4.731
svr           basic              3.027      NaN   3.170  3.155      NaN   0.335  0.429      NaN   5.194  4.815
              full               3.020      NaN   3.215  3.080      NaN   0.310  0.378      NaN   5.294  5.026
              full_aqi           3.018      NaN   3.231  3.124      NaN   0.310  0.377      NaN   5.292  5.029
              full_selected      2.948      NaN   3.156  3.075      NaN   0.325  0.393      NaN   5.233  4.962
              full_weather       2.285      NaN   2.630  2.261      NaN   0.517  0.702      NaN   4.425  3.480
              full_weather_past  2.934      NaN   3.094  2.946      NaN   0.344  0.421      NaN   5.159  4.846
              lag                3.037      NaN   3.281  3.122      NaN   0.307  0.386      NaN   5.304  4.990
xgboost       basic              3.112      NaN   3.707  3.156      NaN   0.178  0.455      NaN   5.775  4.701
              full               3.102      NaN   3.336  3.182      NaN   0.424  0.456      NaN   4.837  4.700
              full_aqi           3.083      NaN   3.416  3.221      NaN   0.401  0.443      NaN   4.930  4.756
              full_selected      3.050      NaN   3.326  3.122      NaN   0.415  0.471      NaN   4.872  4.632
              full_weather       2.3

## Selected hyperparameters

In [5]:
bp = res[(res.variant == "tuned") & (res.stage == "validation")][["model", "feature_config", "best_params", "train_seconds", "n_features_used"]]
display(bp)

,model,feature_config,best_params,train_seconds,n_features_used
2,ridge,basic,"{""model__alpha"": 24.658329458549105}",2.742743,10
6,random_forest,basic,"{""model__n_estimators"": 500, ""model__min_sampl...",35.124420,10
10,xgboost,basic,"{""model__colsample_bytree"": 1.0, ""model__learn...",41.335382,10
14,svr,basic,"{""model__C"": 145.1500859784025, ""model__epsilo...",0.535809,10
18,ensemble,basic,"{""members"": ""random_forest,xgboost,svr"", ""weig...",0.614113,10
22,ridge,lag,"{""model__alpha"": 98.77700294007911}",0.205819,24
26,random_forest,lag,"{""model__n_estimators"": 500, ""model__min_sampl...",36.968891,24
30,xgboost,lag,"{""model__colsample_bytree"": 1.0, ""model__learn...",41.676047,24
34,svr,lag,"{""model__C"": 3.685025727227953, ""model__epsilo...",0.705350,24
38,ensemble,lag,"{""members"": ""random_forest,xgboost,svr"", ""weig...",0.649308,24


## Model selection (validation only)

In [6]:
sel = json.loads((MET / "model_selection.json").read_text())
print(json.dumps(sel, indent=2))

{
  "criterion": "lowest 2024 validation MAE among tuned models and main feature configurations",
  "best": {
    "model": "ensemble",
    "feature_config": "full_weather",
    "validation_mae": 2.2274228180860236
  },
  "per_model": [
    {
      "model": "ensemble",
      "feature_config": "full_weather",
      "validation_mae": 2.2274228180860236
    },
    {
      "model": "random_forest",
      "feature_config": "full_weather",
      "validation_mae": 2.3520385027893407
    },
    {
      "model": "ridge",
      "feature_config": "full_weather",
      "validation_mae": 2.637003530635127
    },
    {
      "model": "svr",
      "feature_config": "full_weather",
      "validation_mae": 2.261124165230498
    },
    {
      "model": "xgboost",
      "feature_config": "full_weather",
      "validation_mae": 2.2489461320231787
    }
  ],
  "periods": {
    "train": "2021-01-01..2023-12-31",
    "validation": "2024-01-01..2024-12-31",
    "test": "2025-01-01..2025-12-31"
  },
  "n_sample